[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/sprints/S11/S11_04_reto_vision_cliente.ipynb)

# S11 · Reto PBL — Visión artificial para el cliente

**Tiempo estimado:** 1 h 45 min de guía en clase (el reto completo son 5 h de PBL) · **Sprint 11**

**Escena del cliente.** TurisData Canarias recibe cada semana cientos de fotos de alojamientos y zonas (las suben los
propietarios y los huéspedes) y hoy las clasifica una persona a mano: *hotel, apartamentos, casa rural, camping, piscina*.
Quiere saber si un modelo podría **pre-clasificarlas** y, sobre todo, **en qué se equivoca** para decidir si se puede fiarse de él.

**Tu entregable:** modelo + demostrador + **análisis de errores** (con ejemplos mal clasificados).

**Dos ramas.**
- **Offline (todo lo ejecutable aquí):** un dataset **sintético** de dibujos esquemáticos, generado por script (no son fotos
  reales) y un modelo con *transfer learning* desde una base preentrenada en una tarea auxiliar.
- **Colab / tu dataset propio (requiere internet o tus fotos):** MobileNetV2 con pesos de ImageNet sobre fotos reales.
  La descarga de pesos **no está verificada** en el taller. El flujo con carpetas propias sí se comprueba con el dataset sintético.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import os, functools, tempfile, shutil
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, f1_score

SEMILLA = 42
keras.utils.set_random_seed(SEMILLA)

## 0. Generador de imágenes sintéticas
Código de apoyo (dibuja con PIL). No hace falta estudiarlo. **Las imágenes que produce no son fotos reales.**
El mismo código está en el script `generar_dataset_sintetico.py` de la carpeta del sprint.

In [ ]:
from PIL import Image, ImageDraw

CLASES_ALOJ = ["hotel", "apartamentos", "casa_rural", "camping", "piscina"]
CLASES_FORMAS = ["rayas_horizontales", "rayas_verticales", "diagonales", "cuadricula", "puntos", "circulo", "rectangulo", "triangulo"]


def _color(rng, base, var=30):
    """Color aleatorio cercano a `base` (tupla RGB)."""
    return tuple(int(np.clip(c + rng.integers(-var, var + 1), 0, 255)) for c in base)


def _final(img, rng, ruido=8):
    """Convierte a uint8 con un poco de ruido y variación de brillo."""
    a = np.asarray(img, dtype="float32") * rng.uniform(0.85, 1.15)
    a = a + rng.normal(0, ruido, a.shape)
    return np.clip(a, 0, 255).astype("uint8")


def _ventanas(d, x0, y0, x1, y1, filas, cols, color, margen):
    """Cuadrícula de ventanas dentro del rectángulo (x0, y0, x1, y1)."""
    ancho, alto = (x1 - x0) / cols, (y1 - y0) / filas
    for f in range(filas):
        for c in range(cols):
            a, b = x0 + c * ancho + margen * ancho, y0 + f * alto + margen * alto
            d.rectangle([a, b, a + ancho * (1 - 2 * margen), b + alto * (1 - 2 * margen)], fill=color)


def dibujar_alojamiento(clase, rng, tam=48, ruido=8):
    """Dibujo esquemático (tam×tam×3, uint8) de un tipo de alojamiento. Datos SINTÉTICOS, no fotos reales."""
    E = 3                                   # supermuestreo para suavizar bordes
    S = tam * E
    img = Image.new("RGB", (S, S), _color(rng, (170, 205, 235), 25))          # cielo
    d = ImageDraw.Draw(img)
    if rng.random() < 0.5:                                                    # sol opcional
        cx, cy, r = rng.uniform(0.1, 0.9) * S, rng.uniform(0.08, 0.25) * S, rng.uniform(0.05, 0.08) * S
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=(250, 235, 120))
    hor = rng.uniform(0.62, 0.72) * S                                         # línea del horizonte
    verde = _color(rng, (120, 170, 90), 25)
    if clase == "piscina":
        d.rectangle([0, hor * 0.6, S, S], fill=_color(rng, (225, 210, 170), 15))   # terraza
        x0, x1 = rng.uniform(0.08, 0.2) * S, rng.uniform(0.75, 0.92) * S
        y0, y1 = rng.uniform(0.5, 0.58) * S, rng.uniform(0.82, 0.94) * S
        d.rectangle([x0 - 3, y0 - 3, x1 + 3, y1 + 3], fill=(245, 245, 245))
        d.rectangle([x0, y0, x1, y1], fill=_color(rng, (60, 150, 210), 20))
        for k in range(3):
            yy = y0 + (k + 1) * (y1 - y0) / 4
            d.line([x0 + 6, yy, x1 - 6, yy], fill=(150, 210, 240), width=2)
        for k in range(2):                                                    # tumbonas
            tx = rng.uniform(0.1, 0.8) * S
            d.rectangle([tx, hor * 0.72, tx + 0.1 * S, hor * 0.72 + 0.04 * S], fill=_color(rng, (240, 120, 60), 30))
    else:
        d.rectangle([0, hor, S, S], fill=verde)                                # césped
    if clase == "hotel":
        w, h = rng.uniform(0.30, 0.42) * S, rng.uniform(0.5, 0.66) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (220, 200, 170)))
        _ventanas(d, x0, y0 + 0.02 * S, x0 + w, hor, int(rng.integers(4, 6)), 3, (60, 90, 140), 0.22)
    elif clase == "apartamentos":
        w, h = rng.uniform(0.7, 0.86) * S, rng.uniform(0.28, 0.38) * S
        x0 = rng.uniform(0.02 * S, 0.98 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 205)))
        _ventanas(d, x0, y0, x0 + w, hor + 0.05 * S, 2, int(rng.integers(5, 7)), (70, 100, 150), 0.25)
        for k in (1, 2):                                                       # balcones (líneas horizontales)
            yy = y0 + k * (hor + 0.05 * S - y0) / 2
            d.line([x0, yy, x0 + w, yy], fill=(120, 110, 100), width=2)
    elif clase == "casa_rural":
        w, h = rng.uniform(0.32, 0.44) * S, rng.uniform(0.2, 0.28) * S
        x0 = rng.uniform(0.1 * S, 0.9 * S - w); y0 = hor - h + 0.05 * S
        d.rectangle([x0, y0, x0 + w, hor + 0.05 * S], fill=_color(rng, (235, 225, 200)))
        d.polygon([(x0 - 0.04 * S, y0), (x0 + w + 0.04 * S, y0), (x0 + w / 2, y0 - rng.uniform(0.14, 0.2) * S)],
                  fill=_color(rng, (170, 70, 50), 25))
        d.rectangle([x0 + w * 0.42, y0 + h * 0.4, x0 + w * 0.58, hor + 0.05 * S], fill=(110, 70, 40))
        d.rectangle([x0 + w * 0.1, y0 + h * 0.2, x0 + w * 0.3, y0 + h * 0.5], fill=(70, 100, 150))
    elif clase == "camping":
        d.rectangle([0, hor - 0.08 * S, S, S], fill=verde)
        w, h = rng.uniform(0.34, 0.5) * S, rng.uniform(0.28, 0.4) * S
        x0 = rng.uniform(0.05 * S, 0.95 * S - w); base = hor + 0.1 * S
        c = _color(rng, [(240, 140, 40), (60, 120, 200), (230, 200, 60)][int(rng.integers(0, 3))], 20)
        d.polygon([(x0, base), (x0 + w, base), (x0 + w / 2, base - h)], fill=c)
        d.polygon([(x0 + w * 0.4, base), (x0 + w * 0.6, base), (x0 + w / 2, base - h * 0.5)], fill=(50, 40, 40))
    if clase != "piscina" and rng.random() < 0.6:                              # árbol de adorno
        tx = rng.choice([rng.uniform(0.03, 0.12), rng.uniform(0.88, 0.97)]) * S
        d.rectangle([tx - 2, hor - 0.02 * S, tx + 2, hor + 0.12 * S], fill=(100, 70, 40))
        d.ellipse([tx - 0.07 * S, hor - 0.16 * S, tx + 0.07 * S, hor + 0.01 * S], fill=(50, 120, 60))
    return _final(img.resize((tam, tam), Image.LANCZOS), rng, ruido)


def dibujar_forma(clase, rng, tam=48):
    """Formas y texturas abstractas (tarea auxiliar de preentrenamiento). Devuelve tam×tam×3 uint8."""
    fondo, tinta = _color(rng, (rng.integers(20, 235),) * 3, 20), _color(rng, tuple(int(v) for v in rng.integers(0, 256, 3)), 10)
    if np.abs(np.array(fondo, dtype=int) - np.array(tinta, dtype=int)).sum() < 150:      # asegura contraste
        tinta = tuple(255 - c for c in fondo)
    E = 2
    S = tam * E
    img = Image.new("RGB", (S, S), fondo)
    d = ImageDraw.Draw(img)
    paso = int(rng.integers(6, 12)) * E
    off = int(rng.integers(0, paso))
    if clase == "rayas_horizontales":
        for y in range(off, S, paso): d.rectangle([0, y, S, y + paso // 2], fill=tinta)
    elif clase == "rayas_verticales":
        for x in range(off, S, paso): d.rectangle([x, 0, x + paso // 2, S], fill=tinta)
    elif clase == "diagonales":
        for k in range(-S, S, paso): d.line([k + off, 0, k + off + S, S], fill=tinta, width=max(2, paso // 3))
    elif clase == "cuadricula":
        for y in range(off, S, paso): d.line([0, y, S, y], fill=tinta, width=2)
        for x in range(off, S, paso): d.line([x, 0, x, S], fill=tinta, width=2)
    elif clase == "puntos":
        r = max(2, paso // 4)
        for y in range(off, S, paso):
            for x in range(off, S, paso): d.ellipse([x - r, y - r, x + r, y + r], fill=tinta)
    else:
        r = rng.uniform(0.18, 0.38) * S
        cx, cy = rng.uniform(r, S - r), rng.uniform(r, S - r)
        if clase == "circulo": d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=tinta)
        elif clase == "rectangulo": d.rectangle([cx - r, cy - r * rng.uniform(0.5, 1), cx + r, cy + r * rng.uniform(0.5, 1)], fill=tinta)
        else: d.polygon([(cx - r, cy + r), (cx + r, cy + r), (cx, cy - r)], fill=tinta)
    return _final(img.resize((tam, tam), Image.LANCZOS), rng)


def generar_lote(funcion, clases, n_por_clase, semilla=42, tam=48):
    """Devuelve (X uint8 [n, tam, tam, 3], y int [n]) con n_por_clase imágenes por clase, mezcladas."""
    rng = np.random.default_rng(semilla)
    X = np.stack([funcion(c, rng, tam) for c in clases for _ in range(n_por_clase)])
    y = np.repeat(np.arange(len(clases)), n_por_clase)
    orden = rng.permutation(len(y))
    return X[orden], y[orden]


PARECIDAS = {"hotel": "apartamentos", "apartamentos": "hotel", "casa_rural": "camping", "camping": "casa_rural", "piscina": "camping"}


def crear_dataset_carpetas(carpeta, n_por_clase=100, semilla=7, tam=48, ruido=8, prop_ambiguas=0.0):
    """Escribe PNG en carpeta/<clase>/img_000.png (estructura que espera image_dataset_from_directory).

    prop_ambiguas: fracción de imágenes cuyo dibujo corresponde a una clase parecida (PARECIDAS) pero se guarda
    en la carpeta de la otra: simula etiquetas dudosas o erróneas. Se anotan en carpeta/_ambiguas.csv (solo docente).
    """
    import os
    rng = np.random.default_rng(semilla)
    ambiguas = []
    for clase in CLASES_ALOJ:
        os.makedirs(os.path.join(carpeta, clase), exist_ok=True)
        for i in range(n_por_clase):
            dibujo = PARECIDAS[clase] if rng.random() < prop_ambiguas else clase
            if dibujo != clase:
                ambiguas.append(f"{clase}/img_{i:03d}.png,{dibujo}")
            Image.fromarray(dibujar_alojamiento(dibujo, rng, tam, ruido)).save(os.path.join(carpeta, clase, f"img_{i:03d}.png"))
    with open(os.path.join(carpeta, "_ambiguas.csv"), "w") as f:
        f.write("archivo,dibujo_real\n" + "\n".join(ambiguas) + "\n")
    return carpeta

## 1. El dataset: una carpeta por clase

`image_dataset_from_directory` espera esta estructura (el nombre de cada carpeta es la etiqueta):

```
datos_sinteticos/alojamientos/
    hotel/           img_000.png  img_001.png ...
    apartamentos/    ...
    casa_rural/      ...
    camping/         ...
    piscina/         ...
```

Para **usar tu propio dataset** (fotos reales con la misma estructura), pon `USAR_DATASET_PROPIO = True` y la ruta de tu carpeta.
Para el sintético, el notebook lo genera solo (60 imágenes por clase, con ruido y un 12 % de imágenes "ambiguas": el dibujo se parece más a otra clase que a la de su carpeta, como pasa con etiquetas dudosas en datos reales) la primera vez.

In [ ]:
USAR_DATASET_PROPIO = False
RUTA_PROPIA = "mis_fotos"                       # carpeta con una subcarpeta por clase
RUTA = RUTA_PROPIA if USAR_DATASET_PROPIO else os.path.join("datos_sinteticos", "alojamientos")

if not USAR_DATASET_PROPIO and not os.path.isdir(RUTA):
    crear_dataset_carpetas(RUTA, n_por_clase=60, semilla=7, tam=48, ruido=40, prop_ambiguas=0.12)
    print("Dataset sintético generado en", RUTA)

**Ejercicio 1 (comprobar el dataset).** Antes de entrenar hay que verificar qué contiene la carpeta. Escribe
`contar_por_clase(ruta)` que devuelva un diccionario `{nombre_de_clase: nº_de_imágenes}` con una entrada por subcarpeta
(cuenta solo archivos `.png`, `.jpg`, `.jpeg`; ignora mayúsculas). Sirve para detectar clases vacías o desequilibradas.

In [ ]:
# TODO: Recorre os.listdir(ruta): para cada subcarpeta cuenta los archivos con esas extensiones
...

conteo = contar_por_clase(RUTA)
print(conteo)
assert len(conteo) >= 2, "Debe haber al menos 2 subcarpetas (clases)"
assert all(v > 0 for v in conteo.values()), "Hay alguna clase sin imágenes"
assert (not USAR_DATASET_PROPIO) <= (conteo == {c: 60 for c in CLASES_ALOJ}), "El dataset sintético debe tener 60 imágenes en cada una de las 5 clases"

### Cargar con `image_dataset_from_directory`
- `validation_split=0.4, subset="both"` devuelve entrenamiento (60 %) y un bloque del 40 % que dividiremos por la mitad en
  **validación** (20 %) y **prueba** (20 %). La misma `seed` garantiza que no se mezclen.
- `image_size` redimensiona todas las imágenes (con fotos reales de distinto tamaño esto es imprescindible).
- El conjunto de validación se devuelve **sin barajar**, así que `file_paths` mantiene el orden y podremos localizar
  cada imagen mal clasificada.

In [ ]:
TAM = 48
ds_ent, ds_resto = keras.utils.image_dataset_from_directory(
    RUTA, validation_split=0.4, subset="both", seed=SEMILLA, image_size=(TAM, TAM), batch_size=32, label_mode="int")
NOMBRES = ds_ent.class_names
N_CLASES = len(NOMBRES)
rutas_resto = ds_resto.file_paths

def a_numpy(ds):
    lotes = list(ds)                  # UNA sola pasada: el entrenamiento se baraja en cada pasada y separar X e y las desalinearía
    X = np.concatenate([x.numpy() for x, _ in lotes]).astype("float32")
    y = np.concatenate([y.numpy() for _, y in lotes]).astype(int)
    return X, y

X_ent, y_ent = a_numpy(ds_ent)
X_resto, y_resto = a_numpy(ds_resto)
mitad = len(X_resto) // 2
X_val, y_val = X_resto[:mitad], y_resto[:mitad]
X_pru, y_pru = X_resto[mitad:], y_resto[mitad:]
rutas_pru = rutas_resto[mitad:]
print("Clases:", NOMBRES)
print(f"entrenamiento {X_ent.shape} · validación {X_val.shape} · prueba {X_pru.shape}")

fig, ejes = plt.subplots(1, 8, figsize=(14, 2.2))
for eje, i in zip(ejes, range(8)):
    eje.imshow(X_ent[i].astype("uint8")); eje.set_title(NOMBRES[y_ent[i]], fontsize=8); eje.axis("off")
plt.suptitle("Ejemplos del entrenamiento"); plt.tight_layout(); plt.show()

## 2. El modelo: *transfer learning* (rama offline)
Igual que en la práctica 2: base convolucional **preentrenada en formas y texturas** (3 200 dibujos, tarea auxiliar) →
se congela → cabeza nueva → *fine-tuning* del último bloque con tasa pequeña. Con fotos reales en Colab, la base sería
MobileNetV2 con pesos de ImageNet (sección 5).

In [ ]:
def construir_base(nombre="base"):
    entrada = keras.layers.Input(shape=(TAM, TAM, 3))
    x = keras.layers.Rescaling(1 / 255)(entrada)
    for i, f in enumerate((16, 32, 64), start=1):
        x = keras.layers.Conv2D(f, 3, padding="same", activation="relu", name=f"bloque{i}_conv")(x)
        x = keras.layers.MaxPooling2D(name=f"bloque{i}_pool")(x)
    x = keras.layers.Flatten(name="aplanar")(x)
    return keras.Model(entrada, x, name=nombre)

def con_cabeza(base, n_clases, dropout=0.3, tasa=1e-3):
    entrada = keras.layers.Input(shape=(TAM, TAM, 3))
    x = keras.layers.Dropout(dropout)(base(entrada))
    salida = keras.layers.Dense(n_clases, activation="softmax", name="cabeza")(x)
    m = keras.Model(entrada, salida)
    m.compile(keras.optimizers.Adam(tasa), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

# Tarea auxiliar: formas y texturas (ejecuta unos 15 s)
X_aux, y_aux = generar_lote(dibujar_forma, CLASES_FORMAS, 400, semilla=3, tam=TAM)
keras.utils.set_random_seed(SEMILLA)
base_pre = construir_base("base_preentrenada")
con_cabeza(base_pre, len(CLASES_FORMAS), dropout=0.0).fit(X_aux, y_aux, epochs=12, batch_size=64, verbose=0)
print("Base preentrenada lista:", base_pre.count_params(), "parámetros")

**Ejercicio 2.** Escribe `modelo_transfer(base_preentrenada, n_clases)` que devuelva un modelo con una **copia congelada**
de la base (usa `keras.models.clone_model` + `set_weights`; después `trainable = False`) y una cabeza nueva con `con_cabeza`.
La base original **no** debe quedar congelada.

In [ ]:
# TODO: clone_model, set_weights(get_weights()), trainable = False; luego con_cabeza(copia, n_clases)
...

_m, _c = modelo_transfer(base_pre, N_CLASES)
assert len(_c.trainable_weights) == 0, "La copia de la base debe estar congelada"
assert len(base_pre.trainable_weights) > 0, "No congeles la base original: trabaja sobre una copia"
assert _m.output_shape == (None, N_CLASES), "La salida debe tener una neurona por clase"

In [ ]:
keras.utils.set_random_seed(SEMILLA)
modelo, base_congelada = modelo_transfer(base_pre, N_CLASES)
modelo.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=30, batch_size=16, verbose=0)
print("Precisión de validación con base congelada:", round(modelo.evaluate(X_val, y_val, verbose=0)[1], 3))

# Fine-tuning del último bloque con tasa pequeña (y recompilando)
for capa in base_congelada.layers:
    capa.trainable = capa.name.startswith("bloque3")
modelo.compile(keras.optimizers.Adam(1e-4), "sparse_categorical_crossentropy", metrics=["accuracy"])
h = modelo.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=25, batch_size=16, verbose=0)
print("Precisión de validación tras fine-tuning:", round(modelo.evaluate(X_val, y_val, verbose=0)[1], 3))

# Referencia: la misma arquitectura entrenada desde cero
keras.utils.set_random_seed(SEMILLA)
desde_cero = con_cabeza(construir_base("desde_cero"), N_CLASES)
desde_cero.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=40, batch_size=16, verbose=0)

## 3. Rendimiento en el conjunto de prueba
Ahora, y solo ahora, se usa la prueba. Además de la precisión global miramos **por clase** (precisión y recall)
y la **matriz de confusión**: dice *qué clases se confunden entre sí*.

In [ ]:
prob_pru = modelo.predict(X_pru, verbose=0)
pred_pru = prob_pru.argmax(axis=1)
pred_cero = desde_cero.predict(X_pru, verbose=0).argmax(axis=1)
print(f"Precisión en prueba → transfer + fine-tuning: {accuracy_score(y_pru, pred_pru):.3f} · desde cero: {accuracy_score(y_pru, pred_cero):.3f}")
print(f"F1 macro → transfer: {f1_score(y_pru, pred_pru, average='macro'):.3f} · desde cero: {f1_score(y_pru, pred_cero, average='macro'):.3f}")
print(classification_report(y_pru, pred_pru, target_names=NOMBRES, zero_division=0))

cm = confusion_matrix(y_pru, pred_pru, labels=range(N_CLASES))
plt.figure(figsize=(5.6, 4.8)); plt.imshow(cm, cmap="Blues")
for i in range(N_CLASES):
    for j in range(N_CLASES):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.xticks(range(N_CLASES), NOMBRES, rotation=30); plt.yticks(range(N_CLASES), NOMBRES)
plt.xlabel("predicho"); plt.ylabel("real"); plt.title("Matriz de confusión (prueba)"); plt.tight_layout(); plt.show()

**Ejercicio 3 (precisión y recall por clase a partir de la matriz).** Con `cm` (filas = real, columnas = predicho), calcula
`recall_clase` (vector: diagonal dividida entre la suma de cada **fila**) y `precision_clase` (diagonal dividida entre la
suma de cada **columna**). Usa `np.diag` y `cm.sum(axis=...)`; si una suma es 0, deja 0. Guarda además en
`clase_mas_dificil` el **nombre** de la clase con menor recall.

In [ ]:
# TODO: np.diag(cm) / cm.sum(axis=1) para el recall; cm.sum(axis=0) para la precisión; np.argmin del recall
...

assert recall_clase.shape == (N_CLASES,) and precision_clase.shape == (N_CLASES,), "Deben ser vectores de una posición por clase"
assert abs(recall_clase.mean() - np.mean([cm[i, i] / max(1, cm[i].sum()) for i in range(N_CLASES)])) < 1e-9, "El recall se divide entre la suma de la FILA"
assert clase_mas_dificil in NOMBRES, "clase_mas_dificil debe ser el nombre de una clase"
print(pd.DataFrame({"recall": recall_clase, "precisión": precision_clase}, index=NOMBRES).round(2))
print("Clase más difícil (menor recall):", clase_mas_dificil)

## 4. Análisis de errores: mirar los ejemplos mal clasificados

Una cifra global oculta **dónde** falla el modelo. Hay que mirar las imágenes concretas: ¿están mal etiquetadas?
¿son borrosas o ruidosas? ¿se parecen a otra clase? Empezamos por los errores **con más confianza** (el modelo estaba
seguro y se equivocó: son los más peligrosos).

**Ejercicio 4.** Escribe `indices_errores(y_real, prob)` que devuelva un array con los índices de las imágenes mal
clasificadas, **ordenados de mayor a menor confianza** de la predicción (la confianza es la probabilidad máxima de cada fila de `prob`).

In [ ]:
# TODO: pred = prob.argmax(1); errores = np.where(pred != y_real)[0]; ordénalos por prob.max(1) descendente
...

_prob = np.array([[0.9, 0.1], [0.4, 0.6], [0.2, 0.8], [0.7, 0.3]])
_res = indices_errores(np.array([1, 1, 0, 0]), _prob)
assert list(_res) == [0, 2], "Los errores son las filas 0 (confianza 0.9) y 2 (0.8); deben salir en ese orden: [0, 2]"
assert len(indices_errores(np.array([0, 1]), np.array([[0.9, 0.1], [0.2, 0.8]]))) == 0, "Sin errores debe devolver un array vacío"

In [ ]:
errores = indices_errores(y_pru, prob_pru)
print(f"{len(errores)} imágenes mal clasificadas de {len(y_pru)}")
n_mostrar = min(10, len(errores))
if n_mostrar:
    fig, ejes = plt.subplots(2, 5, figsize=(13, 6.2))
    for eje in ejes.ravel():
        eje.axis("off")
    for eje, i in zip(ejes.ravel(), errores[:n_mostrar]):
        eje.imshow(X_pru[i].astype("uint8"))
        eje.set_title(f"real: {NOMBRES[y_pru[i]]}\npredicho: {NOMBRES[pred_pru[i]]} ({prob_pru[i].max():.0%})", fontsize=8, color="darkred")
        eje.text(0, -4, os.path.basename(rutas_pru[i]), fontsize=6)
    plt.suptitle("Errores del modelo con más confianza (los primeros son los más preocupantes)"); plt.tight_layout(); plt.show()

In [ ]:
# Qué pares de clases se confunden más
pares = pd.Series([f"{NOMBRES[y_pru[i]]} → {NOMBRES[pred_pru[i]]}" for i in errores]).value_counts()
print(pares.head(5) if len(pares) else "No hay errores en la prueba.")

### Preguntas guía del análisis de errores (para tu informe)
1. ¿Los errores se concentran en una clase o en un par de clases? ¿Tiene sentido (se parecen visualmente)?
2. Mira las imágenes: ¿qué tienen en común (ruido, encuadre, colores, tamaño del objeto)? Formula una **hipótesis**.
3. ¿Hay errores con confianza alta? ¿Qué implica para usar el modelo sin supervisión humana?
4. ¿Qué harías para mejorar: más imágenes de esa clase, *data augmentation*, más *fine-tuning*, otra base preentrenada, revisar etiquetas?

## 5. IA responsable: cuándo el modelo debe pedir ayuda
Un modelo puede **abstenerse**: si su confianza es baja, deriva la foto a una persona. Se mide con dos números:
la **cobertura** (qué fracción de fotos clasifica solo) y la **precisión sobre las que acepta**.

**Ejercicio 5.** Escribe `cobertura_y_precision(prob, y_real, umbral)` que devuelva la tupla `(cobertura, precision_aceptadas)`:
se aceptan las fotos cuya probabilidad máxima sea `>= umbral`. Si no se acepta ninguna, devuelve `(0.0, 0.0)`.

In [ ]:
# TODO: mascara = prob.max(1) >= umbral; cobertura = mascara.mean(); precisión = aciertos entre las aceptadas
...

_p = np.array([[0.9, 0.1], [0.55, 0.45], [0.3, 0.7], [0.6, 0.4]])
_y = np.array([0, 1, 1, 1])
assert cobertura_y_precision(_p, _y, 0.65) == (0.5, 1.0), "Con umbral 0.65 se aceptan 2 de 4 (filas 0 y 2) y ambas aciertan"
assert cobertura_y_precision(_p, _y, 0.99) == (0.0, 0.0), "Si no se acepta ninguna: (0.0, 0.0)"
assert abs(cobertura_y_precision(_p, _y, 0.0)[0] - 1.0) < 1e-9, "Umbral 0: se aceptan todas"

In [ ]:
umbrales = [0.0, 0.5, 0.7, 0.9, 0.95]
tabla = pd.DataFrame([cobertura_y_precision(prob_pru, y_pru, u) for u in umbrales], index=umbrales,
                     columns=["cobertura", "precisión en las aceptadas"]).round(3)
tabla.index.name = "umbral de confianza"
tabla

## 6. Demostrador
Una función que recibe la ruta de **una foto** y devuelve la clase y las probabilidades. En tu entregable la pondrás detrás de
una pequeña interfaz (por ejemplo Gradio o Streamlit en Colab); aquí basta con la función.

In [ ]:
def clasificar_foto(ruta_imagen, modelo_=None, umbral=0.7):
    """Devuelve (clase, confianza, probabilidades por clase) o 'revisar a mano' si la confianza es baja."""
    modelo_ = modelo_ or modelo
    img = keras.utils.load_img(ruta_imagen, target_size=(TAM, TAM))
    p = modelo_.predict(np.expand_dims(keras.utils.img_to_array(img), 0), verbose=0)[0]
    clase = NOMBRES[int(p.argmax())] if p.max() >= umbral else "revisar a mano"
    return clase, float(p.max()), dict(zip(NOMBRES, p.round(3)))

demo_ruta = rutas_pru[0]
print("Foto:", demo_ruta, "· real:", NOMBRES[y_pru[0]])
print(clasificar_foto(demo_ruta))

# Guardar y recargar el modelo (para el demostrador)
carpeta_tmp = tempfile.mkdtemp()
modelo.save(os.path.join(carpeta_tmp, "clasificador_alojamientos.keras"))
recargado = keras.models.load_model(os.path.join(carpeta_tmp, "clasificador_alojamientos.keras"))
assert clasificar_foto(demo_ruta, recargado)[0] == clasificar_foto(demo_ruta)[0], "El modelo recargado debe dar la misma respuesta"
shutil.rmtree(carpeta_tmp, ignore_errors=True)

## 7. Rama Colab: MobileNetV2 con pesos de ImageNet sobre fotos reales

> **AVISO: la parte de descarga de pesos requiere internet y NO se ha verificado en el taller.**
> La construcción del modelo sí se ha comprobado con `weights=None` (sin descargar nada) y el flujo de carpetas
> `image_dataset_from_directory` se ha comprobado con el dataset sintético.

Para fotos reales: pon `USAR_DATASET_PROPIO = True` (sección 1), usa un `TAM` de 96, 128 o 160 y cambia la base por MobileNetV2.
La estructura del entrenamiento es la misma: congelar → cabeza → *fine-tuning* con tasa pequeña → análisis de errores.

In [ ]:
USAR_INTERNET = False        # ← True solo con conexión (Colab)

def construir_mobilenet(n_clases, tam=96, pesos=None):
    """MobileNetV2 (alpha 0.35) congelada + cabeza nueva. pesos='imagenet' descarga los pesos (requiere internet)."""
    base = keras.applications.MobileNetV2(input_shape=(tam, tam, 3), include_top=False, weights=pesos, alpha=0.35)
    base.trainable = False
    entrada = keras.layers.Input(shape=(TAM, TAM, 3))
    x = keras.layers.Resizing(tam, tam)(entrada)
    x = keras.applications.mobilenet_v2.preprocess_input(x)          # [0, 255] → [-1, 1]
    x = keras.layers.GlobalAveragePooling2D()(base(x, training=False))
    x = keras.layers.Dropout(0.3)(x)
    m = keras.Model(entrada, keras.layers.Dense(n_clases, activation="softmax")(x))
    m.compile(keras.optimizers.Adam(1e-3), "sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

mn = construir_mobilenet(N_CLASES, pesos="imagenet" if USAR_INTERNET else None)      # offline: pesos aleatorios, solo estructura
print("MobileNetV2 construida:", mn.count_params(), "parámetros ·", "pesos de ImageNet" if USAR_INTERNET else "pesos aleatorios (sin entrenar)")
if USAR_INTERNET:
    mn.fit(X_ent, y_ent, validation_data=(X_val, y_val), epochs=15, batch_size=16, verbose=0)
    print("MobileNetV2 + ImageNet, precisión en prueba:", round(mn.evaluate(X_pru, y_pru, verbose=0)[1], 3))

## 8. Cómo preparar tu propio dataset (para el reto con fotos reales)
Guía resumida (la versión completa para el docente está en `PREPARAR_DATASET.md`):
- **Estructura:** una carpeta por clase, con nombres sin espacios ni tildes (`hotel`, `casa_rural`…).
- **Tamaño mínimo:** 30 imágenes por clase para probar la mecánica; **100 o más por clase** para resultados presentables;
  clases equilibradas (ninguna con menos de la mitad que la mayor).
- **Calidad:** formatos `.jpg`/`.png`, sin duplicados, sin la misma foto en entrenamiento y prueba; anota de dónde salió cada foto.
- **Licencias:** solo fotos propias, con permiso o con licencia abierta (por ejemplo Creative Commons; guarda autor y licencia);
  sin personas reconocibles (privacidad y RGPD) ni marcas de agua de terceros.

## 9. Tu informe (resumen de una página)
1. Problema, datos (origen, tamaño por clase, licencia) y partición.
2. Modelo y resultados en prueba: precisión, F1 macro, matriz de confusión; comparación con desde cero.
3. **Análisis de errores:** al menos 6 ejemplos mal clasificados, con hipótesis de causa y mejora propuesta.
4. Umbral de confianza recomendado y qué se hace con las fotos que el modelo no acepta.
5. Límites y riesgos: datos sintéticos o no representativos, sesgos (tipos de alojamiento poco frecuentes), privacidad.

## 10. Preguntas de reflexión
1. ¿Qué error es peor para TurisData: confundir "apartamentos" con "hotel" o clasificar un camping como piscina? ¿Cómo lo reflejarías al elegir el umbral?
2. ¿Por qué no basta con la precisión global para aprobar el modelo?
3. Con fotos reales, ¿qué cambiaría respecto a este dataset sintético?

## 11. Resumen: qué has aprendido
- Cargar un dataset de imágenes desde carpetas y separar entrenamiento/validación/prueba sin fugas.
- Entrenar un clasificador con *transfer learning* y compararlo con una base desde cero.
- Evaluar con métricas por clase y matriz de confusión, y **analizar errores** mirando ejemplos concretos.
- Usar la confianza para decidir cuándo el modelo debe pedir ayuda humana.

In [ ]:
# Limpieza (opcional): borra el dataset sintético generado; se puede regenerar cuando quieras con el script
if not USAR_DATASET_PROPIO and os.path.isdir("datos_sinteticos"):
    shutil.rmtree("datos_sinteticos", ignore_errors=True)
    print("Dataset sintético eliminado (no debe subirse al repositorio).")